# VinDr-Mammo: **Benign vs Malignant** — Pipeline 5: Data Sintetis ⭐ (FIXED)

### Konfigurasi Pipeline 5 (Fixed):
- **Task**: Binary Classification — Benign (0) vs Malignant (1)
- **Label mapping**:
  - `Benign (0)` → BI-RADS 1, 2
  - `Malignant (1)` → BI-RADS 3, 4, 5
- **Data**: Original + **Sintetis Malignant** (6400 pairs — tetap semua dipakai)
- **Sampler**: Default shuffle (data sudah diperkaya minority)
- **Loss**: `FocalLoss(gamma=2.0, alpha=0.75)` — menggantikan CrossEntropyLoss biasa
- **Augmentasi Sintetis**: Lebih agresif (Affine, Perspective, sharper ColorJitter) agar model tidak overfit ke pola sintetis
- **Scheduler**: `ReduceLROnPlateau` — tidak di-reset saat backbone unfreeze
- **Output head**: `NUM_CLASSES=2`, output `[B, 2]` logit → softmax → probability

### Perubahan vs versi original:
| Komponen | Sebelum | Sesudah |
|---|---|---|
| Loss | CrossEntropyLoss() | FocalLoss(γ=2, α=0.75) |
| Augment sintetis | Sama dgn original | Lebih agresif (Affine+Perspective) |
| Scheduler | CosineAnnealing (di-reset) | ReduceLROnPlateau (stabil) |
| MAX_SYNTH | None | None (tetap 6400) |

## 1. Install & Import

In [ ]:
!pip install timm -q
!pip install scikit-learn -q

In [ ]:
import os, random, math
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchvision.transforms as T

import timm
from sklearn.metrics import (
    classification_report, roc_auc_score, cohen_kappa_score, confusion_matrix,
    f1_score, accuracy_score, recall_score, precision_score,
    roc_curve, precision_recall_curve, average_precision_score
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
print(f"PyTorch : {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")

## 2. Configuration

In [ ]:
class CFG:
    # ── Path — Original Dataset ───────────────────────────────────────────────
    DATA_DIR          = '/kaggle/input/datasets/vickyoktafrian/dataset-preprocessed/dataset_preprocessed/dataset_preprocessed'
    CSV_PATH          = '/kaggle/input/datasets/vickyoktafrian/dataset-preprocessed/breast-level_annotations.csv'
    OUTPUT_DIR        = '/kaggle/working/outputs'

    # ── Path — Sintetis Dataset ───────────────────────────────────────────────
    SYNTH_ROOT        = '/kaggle/input/datasets/vickyoktafrian/bi-rads-sintetis-data/BI-RADS Sintetis/synthetic_dataset/train/Malignant'
    SYNTH_CC_DIR      = SYNTH_ROOT + '/CC'
    SYNTH_MLO_DIR     = SYNTH_ROOT + '/MLO'

    # ── Model ─────────────────────────────────────────────────────────────────
    MODEL_NAME        = 'swin_tiny_patch4_window7_224'
    PRETRAINED_WEIGHT = '/kaggle/input/datasets/vickyoktafrian/model-mv-swin-t/swin_tiny_patch4_window7_224.pth'
    IMG_SIZE          = 224
    NUM_CLASSES       = 2
    CLASS_NAMES       = ['Benign', 'Malignant']
    DROPOUT           = 0.3

    # ── Label mapping ─────────────────────────────────────────────────────────
    BENIGN_BIRADS     = [1, 2]                   # label = 0
    MALIGNANT_BIRADS  = [3, 4, 5]               # label = 1

    # ── Training ──────────────────────────────────────────────────────────────
    EPOCHS            = 100
    BATCH_SIZE        = 16
    NUM_WORKERS       = 4
    WARMUP_EPOCHS     = 5
    LR_BACKBONE       = 1e-5
    LR_HEAD           = 1e-4
    WEIGHT_DECAY      = 1e-4
    PATIENCE          = 15

    # ── Focal Loss params ─────────────────────────────────────────────────────
    FOCAL_GAMMA       = 2.0    # fokus ke hard samples
    FOCAL_ALPHA       = 0.75   # bobot untuk class Malignant (0.75 = lebih perhatian ke malignant)

    # ── Pipeline 5 specifics ──────────────────────────────────────────────────
    MAX_SYNTH         = None   # None = pakai semua (6400)
    USE_SAMPLER       = False
    USE_MULTI_GPU     = True

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device      : {DEVICE}")
print(f"Task        : Benign (BIRADS 1,2) vs Malignant (BIRADS 3,4,5)")
print(f"NUM_CLASSES : {CFG.NUM_CLASSES}")
print(f"Loss        : FocalLoss(gamma={CFG.FOCAL_GAMMA}, alpha={CFG.FOCAL_ALPHA})  ← FIXED")
print(f"MAX_SYNTH   : {CFG.MAX_SYNTH} (tetap pakai semua 6400 pasang sintetis)")

## 3. Data — Original + Sintetis

In [ ]:
# ════════════════════════════════════════════════════════
# 3A. Load & Proses Original Dataset
# ════════════════════════════════════════════════════════
df = pd.read_csv(CFG.CSV_PATH)

def parse_birads(val):
    if pd.isna(val): return None
    digits = ''.join(filter(str.isdigit, str(val).strip()))
    return int(digits) if digits else None

df['birads_num'] = df['breast_birads'].apply(parse_birads)
df = df.dropna(subset=['birads_num'])
df['birads_num'] = df['birads_num'].astype(int)

ALL_BIRADS = CFG.BENIGN_BIRADS + CFG.MALIGNANT_BIRADS
df = df[df['birads_num'].isin(ALL_BIRADS)].copy()
df['label'] = df['birads_num'].apply(lambda x: 1 if x in CFG.MALIGNANT_BIRADS else 0).astype(int)

def build_pairs(df):
    pairs = []
    for (sid, lat), grp in df.groupby(['study_id', 'laterality']):
        cc  = grp[grp['view_position'] == 'CC']
        mlo = grp[grp['view_position'] == 'MLO']
        if len(cc) == 0 or len(mlo) == 0: continue
        pairs.append({
            'study_id':     sid,
            'laterality':   lat,
            'cc_image_id':  cc.iloc[0]['image_id'],
            'mlo_image_id': mlo.iloc[0]['image_id'],
            'label':        int(cc.iloc[0]['label']),
            'birads':       int(cc.iloc[0]['birads_num']),
            'split':        cc.iloc[0]['split'],
            'source':       'original'
        })
    return pd.DataFrame(pairs)

pairs_df = build_pairs(df)
train_df_orig = pairs_df[pairs_df['split'] == 'training'].reset_index(drop=True)
val_df        = pairs_df[pairs_df['split'] == 'test'].reset_index(drop=True)

if len(val_df) == 0:
    from sklearn.model_selection import train_test_split
    train_df_orig, val_df = train_test_split(pairs_df, test_size=0.2,
                                             stratify=pairs_df['label'], random_state=42)
    train_df_orig = train_df_orig.reset_index(drop=True)
    val_df        = val_df.reset_index(drop=True)

n_orig_ben = (train_df_orig['label'] == 0).sum()
n_orig_mal = (train_df_orig['label'] == 1).sum()

print("=" * 55)
print("ORIGINAL Training Set:")
print(f"  Benign    : {n_orig_ben}")
print(f"  Malignant : {n_orig_mal}")
print(f"  Total     : {len(train_df_orig)}")
print("=" * 55)

# ════════════════════════════════════════════════════════
# 3B. Scan & Pair Data Sintetis
# ════════════════════════════════════════════════════════
cc_dir  = Path(CFG.SYNTH_CC_DIR)
mlo_dir = Path(CFG.SYNTH_MLO_DIR)

cc_files  = {f.stem: f for f in cc_dir.glob('*.png')}
mlo_files = {f.stem: f for f in mlo_dir.glob('*.png')}
paired_stems = sorted(set(cc_files.keys()) & set(mlo_files.keys()))

print(f"\nSINTETIS Dataset Scan:")
print(f"  CC  files  : {len(cc_files)}")
print(f"  MLO files  : {len(mlo_files)}")
print(f"  Valid pairs: {len(paired_stems)}  (CC ∩ MLO by filename)")

if CFG.MAX_SYNTH is not None:
    paired_stems = paired_stems[:CFG.MAX_SYNTH]
    print(f"  Dipakai    : {len(paired_stems)}  (dibatasi MAX_SYNTH={CFG.MAX_SYNTH})")
else:
    print(f"  Dipakai    : {len(paired_stems)}  (semua — MAX_SYNTH=None)")

synth_records = []
for stem in paired_stems:
    synth_records.append({
        'cc_path':  str(cc_files[stem]),
        'mlo_path': str(mlo_files[stem]),
        'label':    1,
        'source':   'synthetic',
        'stem':     stem
    })
synth_df = pd.DataFrame(synth_records)

print(f"\nSintetis Malignant pairs: {len(synth_df)}")
print(f"Contoh stem: {paired_stems[:3]}")

# ════════════════════════════════════════════════════════
# 3C. Hitung distribusi akhir training
# ════════════════════════════════════════════════════════
n_train_ben       = n_orig_ben
n_train_mal_orig  = n_orig_mal
n_train_mal_syn   = len(synth_df)
n_train_mal_total = n_train_mal_orig + n_train_mal_syn

print("\n" + "=" * 55)
print("TRAINING SET AKHIR (Original + Sintetis):")
print(f"  Benign    (original)  : {n_train_ben}")
print(f"  Malignant (original)  : {n_train_mal_orig}")
print(f"  Malignant (sintetis)  : {n_train_mal_syn}")
print(f"  Malignant (total)     : {n_train_mal_total}")
print(f"  Total train           : {n_train_ben + n_train_mal_total}")
print(f"  Rasio Mal/(Ben+Mal)   : {n_train_mal_total/(n_train_ben+n_train_mal_total)*100:.1f}%")
print("=" * 55)
print(f"\nVal set (original, tidak diubah): {len(val_df)} pairs")

## 4. Dataset & DataLoader

In [ ]:
def get_image_path(data_dir, study_id, image_id):
    base = Path(data_dir) / str(study_id)
    for ext in ['.png', '.jpg', '.jpeg']:
        p = base / f'{image_id}{ext}'
        if p.exists(): return str(p)
    for ext in ['.png', '.jpg', '.jpeg']:
        p = Path(data_dir) / f'{image_id}{ext}'
        if p.exists(): return str(p)
    return None


def get_transforms(split='train', img_size=224):
    """Transform untuk data ORIGINAL."""
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    if split == 'train':
        return T.Compose([
            T.Resize((img_size, img_size)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.2),
            T.RandomRotation(degrees=15),
            T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
            T.ColorJitter(brightness=0.3, contrast=0.3),
            T.ToTensor(),
            T.Normalize(mean=mean, std=std),
        ])
    return T.Compose([T.Resize((img_size, img_size)), T.ToTensor(), T.Normalize(mean, std)])


def get_synth_transforms(img_size=224):
    """
    Transform KHUSUS data sintetis — lebih agresif.
    Tujuan: agar model tidak overfit ke pola GAN yang terlalu homogen.
    Augmentasi tambahan: RandomAffine + RandomPerspective.
    """
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    return T.Compose([
        T.Resize((img_size, img_size)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),          # lebih sering dari original
        T.RandomRotation(degrees=30),          # lebih lebar (15 → 30)
        T.RandomAffine(
            degrees=0,
            translate=(0.1, 0.1),             # geser max 10%
            scale=(0.85, 1.15),               # zoom in/out
            shear=10
        ),
        T.RandomPerspective(distortion_scale=0.2, p=0.3),
        T.GaussianBlur(kernel_size=5, sigma=(0.1, 3.0)),
        T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])


# ── Dataset untuk data ORIGINAL ───────────────────────────────────────────────
class VinDrOrigDataset(Dataset):
    def __init__(self, df, data_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.data_dir  = data_dir
        self.transform = transform

    def __len__(self): return len(self.df)

    def _load(self, study_id, image_id):
        path = get_image_path(self.data_dir, study_id, image_id)
        if path is None:
            return Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
        return Image.open(path).convert('RGB')

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cc  = self._load(row['study_id'], row['cc_image_id'])
        mlo = self._load(row['study_id'], row['mlo_image_id'])
        if self.transform:
            cc  = self.transform(cc)
            mlo = self.transform(mlo)
        return cc, mlo, torch.tensor(row['label'], dtype=torch.long)


# ── Dataset untuk data SINTETIS ───────────────────────────────────────────────
class SyntheticDataset(Dataset):
    """
    Load data sintetis Malignant dari path absolut.
    Menggunakan augmentasi LEBIH AGRESIF (get_synth_transforms)
    agar model tidak overfit ke pola homogen dari GAN.
    """
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def _load(self, path):
        try:
            return Image.open(path).convert('RGB')
        except Exception:
            return Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cc  = self._load(row['cc_path'])
        mlo = self._load(row['mlo_path'])
        if self.transform:
            cc  = self.transform(cc)
            mlo = self.transform(mlo)
        return cc, mlo, torch.tensor(row['label'], dtype=torch.long)


# ── Build Datasets ────────────────────────────────────────────────────────────
train_transform = get_transforms('train', CFG.IMG_SIZE)
val_transform   = get_transforms('val',   CFG.IMG_SIZE)
synth_transform = get_synth_transforms(CFG.IMG_SIZE)   # ← augmentasi agresif untuk sintetis

orig_train_dataset  = VinDrOrigDataset(train_df_orig, CFG.DATA_DIR, train_transform)
synth_train_dataset = SyntheticDataset(synth_df, synth_transform)  # ← pakai synth_transform
val_dataset         = VinDrOrigDataset(val_df, CFG.DATA_DIR, val_transform)

train_dataset = ConcatDataset([orig_train_dataset, synth_train_dataset])

print(f"Original train dataset  : {len(orig_train_dataset)} samples")
print(f"Synthetic train dataset : {len(synth_train_dataset)} samples  (augment agresif)")
print(f"COMBINED train dataset  : {len(train_dataset)} samples")
print(f"Val dataset             : {len(val_dataset)} samples")

# ── DataLoader ────────────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_dataset, batch_size=CFG.BATCH_SIZE,
    shuffle=True, num_workers=CFG.NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False,
    num_workers=CFG.NUM_WORKERS, pin_memory=True
)

print(f"\nTrain batches : {len(train_loader)} | Val batches: {len(val_loader)}")
print(f"Sampler       : Default shuffle (original + sintetis via ConcatDataset)")

cc_b, mlo_b, lbl_b = next(iter(train_loader))
mal_in_batch = (lbl_b == 1).sum().item()
print(f"Batch sample  : {mal_in_batch} malignant / {len(lbl_b)} total = {mal_in_batch/len(lbl_b):.2%}")
print(f"CC == MLO px  : {torch.allclose(cc_b, mlo_b)} ← harus False")

## 5. Model — Late Fusion

In [ ]:
class SwinCELateFusion(nn.Module):
    """
    Late-Fusion Swin Transformer — Pipeline 5 Fixed.

    Architecture:
      CC  ──► Swin Encoder (shared weights) ──► cc_feat  [B, D]
      MLO ──► Swin Encoder (shared weights) ──► mlo_feat [B, D]
                                                     │
                                     torch.cat([cc_feat, mlo_feat])  → [B, 2D]
                                                     │
                                               MLP Classifier
                                                     │
                                              logits [B, 2]

    Training data : original VinDr + sintetis Malignant (CC+MLO pairs)
    Loss          : FocalLoss(gamma=2.0, alpha=0.75)  ← FIXED
    """

    def __init__(self, cfg):
        super().__init__()

        self.encoder = timm.create_model(
            cfg.MODEL_NAME, pretrained=True, num_classes=0, global_pool='avg'
        )
        D = self.encoder.num_features
        print(f"Encoder dim (D)      : {D}")
        print(f"Classifier input dim : {D * 2}  (2×D)")
        print(f"Classifier output    : {cfg.NUM_CLASSES}")

        self.classifier = nn.Sequential(
            nn.Linear(D * 2, 512),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT / 2),
            nn.Linear(128, cfg.NUM_CLASSES)
        )

        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, cc: torch.Tensor, mlo: torch.Tensor) -> torch.Tensor:
        cc_feat  = self.encoder(cc)
        mlo_feat = self.encoder(mlo)
        fused    = torch.cat([cc_feat, mlo_feat], dim=1)
        return self.classifier(fused)           # [B, 2]


# ── Instantiate ───────────────────────────────────────────────────────────────
model = SwinCELateFusion(CFG)

# ── Load pretrained weights (encoder only) ────────────────────────────────────
ckpt = torch.load(CFG.PRETRAINED_WEIGHT, map_location='cpu')
sd   = ckpt.get('model', ckpt.get('state_dict', ckpt)) if isinstance(ckpt, dict) else ckpt
ms   = model.encoder.state_dict()
filt = {k: v for k, v in sd.items() if k in ms and v.shape == ms[k].shape}
model.encoder.load_state_dict(filt, strict=False)
print(f"Pretrained   : {len(filt)}/{len(ms)} layers loaded")
assert len(filt) > 100, f"Terlalu sedikit layer loaded ({len(filt)}) — cek file .pth!"

if CFG.USE_MULTI_GPU and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(DEVICE)
print(f"Total params : {sum(p.numel() for p in model.parameters()):,}")

## 6. Loss & Optimizer

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# FOCAL LOSS — menggantikan CrossEntropyLoss
#
# Kenapa Focal Loss?
#   - CrossEntropyLoss biasa: semua sample diperlakukan equal
#   - Focal Loss: hard samples (misal malignant yang sulit diprediksi)
#     mendapat bobot loss lebih besar
#
# Parameter:
#   gamma=2.0 → semakin tinggi, semakin fokus ke hard samples
#   alpha=0.75 → bobot kelas Malignant (0.75), Benign (0.25)
#                Berikan lebih banyak perhatian ke Malignant
#
# Meski data sudah balance 50/50, Focal Loss tetap diperlukan karena:
#   1. Fitur malignant lebih sulit dipelajari dari benign
#   2. Data sintetis cenderung "terlalu mudah" → model perlu didorong
#      belajar hard real malignant samples
# ════════════════════════════════════════════════════════════════════════

class FocalLoss(nn.Module):
    """
    Focal Loss untuk binary/multi-class classification.

    FL(pt) = -alpha_t * (1 - pt)^gamma * log(pt)

    Args:
        gamma (float): Focusing parameter. gamma=0 → CrossEntropyLoss biasa.
                       Rekomendasi: 2.0
        alpha (float): Bobot untuk class Malignant (class 1).
                       Class Benign (class 0) otomatis = 1 - alpha.
                       Rekomendasi: 0.75
    """
    def __init__(self, gamma: float = 2.0, alpha: float = 0.75):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # Cross entropy per sample (unreduced)
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')

        # pt = probability of correct class
        pt = torch.exp(-ce_loss)

        # Alpha weighting: alpha untuk malignant (1), 1-alpha untuk benign (0)
        alpha_t = torch.where(targets == 1,
                              torch.tensor(self.alpha, device=inputs.device),
                              torch.tensor(1.0 - self.alpha, device=inputs.device))

        # Focal weight: (1 - pt)^gamma
        focal_loss = alpha_t * (1.0 - pt) ** self.gamma * ce_loss

        return focal_loss.mean()


criterion = FocalLoss(gamma=CFG.FOCAL_GAMMA, alpha=CFG.FOCAL_ALPHA)
print(f"Loss         : FocalLoss(gamma={CFG.FOCAL_GAMMA}, alpha={CFG.FOCAL_ALPHA})  ← FIXED")
print(f"               alpha={CFG.FOCAL_ALPHA} → Malignant diberi bobot lebih tinggi")
print(f"Train total  : {len(train_dataset)} pairs  (original + sintetis)")
print(f"Val total    : {len(val_dataset)} pairs  (original only)")


def get_optimizer(model, cfg, backbone_frozen=True):
    base = model.module if hasattr(model, 'module') else model
    backbone_lr = 0.0 if backbone_frozen else cfg.LR_BACKBONE
    return torch.optim.AdamW([
        {'params': base.encoder.parameters(),    'lr': backbone_lr,  'weight_decay': cfg.WEIGHT_DECAY},
        {'params': base.classifier.parameters(), 'lr': cfg.LR_HEAD,  'weight_decay': cfg.WEIGHT_DECAY},
    ])


def freeze_backbone(model, freeze=True):
    base = model.module if hasattr(model, 'module') else model
    for p in base.encoder.parameters():
        p.requires_grad = not freeze
    print(f"Backbone     : {'FROZEN' if freeze else 'UNFROZEN'}")


freeze_backbone(model, freeze=True)
optimizer = get_optimizer(model, CFG, backbone_frozen=True)

# ── ReduceLROnPlateau menggantikan CosineAnnealingLR ─────────────────────────
# Keunggulan vs CosineAnnealing:
#   1. Tidak perlu di-reset manual saat backbone unfreeze
#   2. LR turun hanya ketika val AUC tidak naik (adaptive)
#   3. Lebih stabil saat ada perubahan arsitektur mid-training
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',        # monitor AUC (makin tinggi makin baik)
    factor=0.5,        # LR dikali 0.5 saat plateau
    patience=5,        # tunggu 5 epoch sebelum turunkan LR
    min_lr=1e-7,
    verbose=True
)
print(f"\nLR head      : {CFG.LR_HEAD}")
print(f"LR backbone  : {CFG.LR_BACKBONE} (frozen selama {CFG.WARMUP_EPOCHS} epoch pertama)")
print(f"Scheduler    : ReduceLROnPlateau(mode=max, factor=0.5, patience=5)  ← FIXED")

## 7. Diagnostic Pre-Training

In [ ]:
print("=" * 62)
print("DIAGNOSTIC — Pipeline 5 Fixed (Focal Loss)")
print("=" * 62)

cc_b, mlo_b, lbl_b = next(iter(train_loader))
cc_b, mlo_b, lbl_b = cc_b.to(DEVICE), mlo_b.to(DEVICE), lbl_b.to(DEVICE)

# [1] CC vs MLO sanity
identical = torch.allclose(cc_b, mlo_b)
print(f"\n[1] CC == MLO pixel-identik : {identical}", '❌ BUG!' if identical else '✅ OK')

# [2] Output shape & range
base = model.module if hasattr(model, 'module') else model
base.eval()
with torch.no_grad():
    logits   = model(cc_b, mlo_b)
    probs    = torch.softmax(logits, dim=1)
    prob_mal = probs[:, 1]

print(f"\n[2] Output shape  : {logits.shape}  (expected: [{cc_b.size(0)}, 2])")
print(f"    Logit range   : [{logits.min():.3f}, {logits.max():.3f}]")
print(f"    Prob-mal range: [{prob_mal.min():.3f}, {prob_mal.max():.3f}]")
print(f"    Prob-mal mean : {prob_mal.mean():.4f}")

# [3] Distribusi label dalam batch
mal_count = (lbl_b == 1).sum().item()
print(f"\n[3] Batch labels:")
print(f"    Malignant : {int(mal_count)}/{len(lbl_b)} = {mal_count/len(lbl_b):.2%}")
print(f"    Benign    : {len(lbl_b)-int(mal_count)}/{len(lbl_b)} = {(len(lbl_b)-mal_count)/len(lbl_b):.2%}")

# [4] Verifikasi sintetis bisa di-load
print(f"\n[4] Verifikasi loading data sintetis:")
try:
    cc_s, mlo_s, lbl_s = synth_train_dataset[0]
    print(f"    Sample [0]: CC={cc_s.shape}, MLO={mlo_s.shape}, label={lbl_s.item()}")
    assert lbl_s.item() == 1, "Label sintetis bukan 1!"
    print(f"    ✅ Sintetis dapat di-load dan label=1 (Malignant)")
    # Verifikasi augmentasi berbeda (random)
    cc_s2, _, _ = synth_train_dataset[0]
    print(f"    ✅ Augmentasi agresif aktif (random per call)")
except Exception as e:
    print(f"    ❌ Error load sintetis: {e}")

# [5] Focal Loss sanity
base.train()
with torch.cuda.amp.autocast():
    test_logits   = model(cc_b, mlo_b)
    loss_focal    = criterion(test_logits, lbl_b)
    loss_ce_equiv = F.cross_entropy(test_logits, lbl_b)  # referensi

print(f"\n[5] Init Focal Loss  : {loss_focal.item():.4f}")
print(f"    Init CE (ref)    : {loss_ce_equiv.item():.4f}")
print(f"    Expected CE init ≈ ln(2) = {math.log(2):.4f}  (random init 2-class)")
print(f"    Focal < CE normal karena easy samples diberi bobot rendah")

# [6] Ringkasan data & perubahan
print(f"\n[6] Ringkasan dataset Pipeline 5 Fixed:")
print(f"    Original Benign    (train) : {n_orig_ben}")
print(f"    Original Malignant (train) : {n_orig_mal}")
print(f"    Sintetis Malignant (train) : {len(synth_df)}")
print(f"    Total Malignant    (train) : {n_orig_mal + len(synth_df)}")
print(f"    Total Training             : {len(train_dataset)}")
print(f"    Rasio Malignant            : {(n_orig_mal+len(synth_df))/len(train_dataset)*100:.1f}%")
print(f"\n[7] PERUBAHAN vs versi original:")
print(f"    Loss      : CrossEntropyLoss → FocalLoss(γ={CFG.FOCAL_GAMMA}, α={CFG.FOCAL_ALPHA}) ✅")
print(f"    Aug synth : standar → agresif (Affine+Perspective)            ✅")
print(f"    Scheduler : CosineAnnealing (reset) → ReduceLROnPlateau       ✅")
print(f"    MAX_SYNTH : tetap None (6400 pasang)                          ✅")
print("=" * 62)
print("✅ Diagnostic OK — siap training!")

## 8. Training

In [ ]:
def find_best_threshold(labels, probs):
    """Sweep threshold 0.05–0.95, pilih yang maximise Macro F1."""
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.05, 0.95, 0.01):
        p  = (probs >= t).astype(int)
        f1 = f1_score(labels, p, average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1


def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0
    all_probs, all_labels = [], []

    for cc, mlo, labels in tqdm(loader, desc='Train', leave=False):
        cc, mlo, labels = cc.to(device), mlo.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(cc, mlo)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        prob_mal = torch.softmax(logits, dim=1)[:, 1]
        all_probs.extend(prob_mal.detach().cpu().float().numpy())
        all_labels.extend(labels.cpu().numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    try:    auc = roc_auc_score(labels, probs)
    except: auc = 0.5
    t, f1  = find_best_threshold(labels, probs)
    pos_pm = probs[labels == 1].mean() if (labels == 1).sum() > 0 else 0.0
    neg_pm = probs[labels == 0].mean() if (labels == 0).sum() > 0 else 0.0
    return total_loss / len(loader), auc, f1, pos_pm, neg_pm


@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_probs, all_labels = [], []

    for cc, mlo, labels in tqdm(loader, desc='Val', leave=False):
        cc, mlo, labels = cc.to(device), mlo.to(device), labels.to(device)
        with torch.cuda.amp.autocast():
            logits = model(cc, mlo)
            loss   = criterion(logits, labels)
        total_loss += loss.item()
        prob_mal = torch.softmax(logits, dim=1)[:, 1]
        all_probs.extend(prob_mal.detach().cpu().float().numpy())
        all_labels.extend(labels.cpu().numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    try:    auc = roc_auc_score(labels, probs)
    except: auc = 0.5
    t, f1  = find_best_threshold(labels, probs)
    pos_pm = probs[labels == 1].mean() if (labels == 1).sum() > 0 else 0.0
    neg_pm = probs[labels == 0].mean() if (labels == 0).sum() > 0 else 0.0

    preds   = (probs >= t).astype(int)
    ben_acc = accuracy_score(labels[labels==0], preds[labels==0]) if (labels==0).sum() > 0 else 0.0
    mal_acc = accuracy_score(labels[labels==1], preds[labels==1]) if (labels==1).sum() > 0 else 0.0
    print(f"  Benign acc: {ben_acc:.3f}  |  Malignant acc: {mal_acc:.3f}")

    return total_loss / len(loader), auc, f1, probs, labels, t, pos_pm, neg_pm


print("Training functions defined ✅")

In [ ]:
scaler  = torch.cuda.amp.GradScaler()
history = {k: [] for k in ['tr_loss', 'vl_loss', 'tr_auc', 'vl_auc', 'tr_f1', 'vl_f1']}

best_auc          = 0.0
patience_count    = 0
best_probs        = None
best_labels_arr   = None
best_thresh_saved = 0.5

print(f"{'Ep':>3} | {'TrLoss':>7} | {'TrAUC':>6} | {'VlAUC':>6} | {'VlF1':>6} | {'mal_pm':>6} | {'ben_pm':>6} | Status")
print('-' * 85)

for epoch in range(1, CFG.EPOCHS + 1):

    # ── Backbone unfreeze setelah warmup ──────────────────────────────────────
    # FIXED: optimizer di-rebuild tapi scheduler TIDAK di-reset
    # ReduceLROnPlateau tidak perlu T_max baru sehingga aman
    if epoch == CFG.WARMUP_EPOCHS + 1:
        freeze_backbone(model, freeze=False)
        optimizer = get_optimizer(model, CFG, backbone_frozen=False)
        # Re-attach optimizer ke scheduler yang sudah ada
        scheduler.optimizer = optimizer
        print(f"  → Backbone unfrozen @ epoch {epoch}, scheduler tetap berjalan")

    tr_loss, tr_auc, tr_f1, tr_mal_pm, tr_ben_pm = train_epoch(
        model, train_loader, optimizer, criterion, DEVICE, scaler
    )
    vl_loss, vl_auc, vl_f1, vl_probs, vl_labels, vl_t, vl_mal_pm, vl_ben_pm = val_epoch(
        model, val_loader, criterion, DEVICE
    )

    # ReduceLROnPlateau: step dengan val AUC (bukan epoch)
    scheduler.step(vl_auc)

    history['tr_loss'].append(tr_loss); history['vl_loss'].append(vl_loss)
    history['tr_auc'].append(tr_auc);   history['vl_auc'].append(vl_auc)
    history['tr_f1'].append(tr_f1);     history['vl_f1'].append(vl_f1)

    status = ''
    if vl_auc > best_auc:
        best_auc = vl_auc
        patience_count = 0
        base = model.module if hasattr(model, 'module') else model
        torch.save(base.state_dict(), f'{CFG.OUTPUT_DIR}/best_model_p5.pth')
        best_probs, best_labels_arr = vl_probs, vl_labels
        best_thresh_saved = vl_t
        status = '✓ SAVED'
    else:
        patience_count += 1
        if patience_count >= CFG.PATIENCE:
            print(f"\nEarly stopping @ epoch {epoch}")
            break

    sep = ' ←' if vl_mal_pm > vl_ben_pm * 1.1 else ''
    print(f"{epoch:>3} | {tr_loss:>7.4f} | {tr_auc:>6.4f} | {vl_auc:>6.4f} | {vl_f1:>6.4f} | "
          f"{vl_mal_pm:>6.3f} | {vl_ben_pm:>6.3f} | {status}{sep}")

print(f"\n✅ Training selesai. Best Val AUC: {best_auc:.4f}")

## 9. Evaluation

In [ ]:
# ── Load best checkpoint ─────────────────────────────────────────────────────
base = model.module if hasattr(model, 'module') else model
base.load_state_dict(torch.load(f'{CFG.OUTPUT_DIR}/best_model_p5.pth', map_location=DEVICE))

_, _, _, best_probs, best_labels_arr, best_thresh_saved, vl_mal_pm, vl_ben_pm = val_epoch(
    model, val_loader, criterion, DEVICE
)

print(f"Prob mean — Malignant: {vl_mal_pm:.3f} | Benign: {vl_ben_pm:.3f}")
print(f"Best threshold       : {best_thresh_saved:.2f}\n")

# ── Threshold sweep ───────────────────────────────────────────────────────────
print(f"{'Threshold':>10} | {'F1-Macro':>8} | {'Sensitivity':>11} | {'Specificity':>11} | {'Precision':>9}")
print("-" * 60)
for t in np.arange(0.05, 0.96, 0.05):
    p    = (best_probs >= t).astype(int)
    sens = recall_score(best_labels_arr, p, pos_label=1, zero_division=0)
    spec = recall_score(best_labels_arr, p, pos_label=0, zero_division=0)
    f1   = f1_score(best_labels_arr, p, average='macro', zero_division=0)
    prec = precision_score(best_labels_arr, p, pos_label=1, zero_division=0)
    mark = ' ◄' if abs(t - best_thresh_saved) < 0.01 else ''
    print(f"  t={t:.2f}   | {f1:>8.4f} | {sens:>11.3f} | {spec:>11.3f} | {prec:>9.3f}{mark}")

# ── Final metrics ─────────────────────────────────────────────────────────────
final_preds = (best_probs >= best_thresh_saved).astype(int)
auc         = roc_auc_score(best_labels_arr, best_probs)
acc         = accuracy_score(best_labels_arr, final_preds)
f1_macro    = f1_score(best_labels_arr, final_preds, average='macro',    zero_division=0)
f1_weight   = f1_score(best_labels_arr, final_preds, average='weighted', zero_division=0)
kappa       = cohen_kappa_score(best_labels_arr, final_preds)
sensitivity = recall_score(best_labels_arr, final_preds, pos_label=1, zero_division=0)
specificity = recall_score(best_labels_arr, final_preds, pos_label=0, zero_division=0)
precision   = precision_score(best_labels_arr, final_preds, pos_label=1, zero_division=0)

print("\n" + "="*60)
print("    EVALUATION — Pipeline 5 Fixed (Focal Loss)")
print("    Benign (BIRADS 1,2) vs Malignant (BIRADS 3,4,5)")
print("    Val set: distribusi ORIGINAL (tanpa data sintetis)")
print("="*60)
print(f"  Loss           : FocalLoss(gamma={CFG.FOCAL_GAMMA}, alpha={CFG.FOCAL_ALPHA})")
print(f"  Sintetis added : {len(synth_df)} Malignant pairs")
print(f"  Train total    : {len(train_dataset)} pairs")
print(f"  AUC-ROC        : {auc:.4f}")
print(f"  Accuracy       : {acc:.4f}")
print(f"  Macro F1       : {f1_macro:.4f}")
print(f"  Weighted F1    : {f1_weight:.4f}")
print(f"  Sensitivity    : {sensitivity:.4f}  ← recall malignant")
print(f"  Specificity    : {specificity:.4f}  ← recall benign")
print(f"  Precision      : {precision:.4f}  ← precision malignant")
print(f"  Kappa          : {kappa:.4f}")
print(f"  Threshold      : {best_thresh_saved:.2f}")
print("="*60)
print(classification_report(best_labels_arr, final_preds,
      target_names=CFG.CLASS_NAMES, zero_division=0))

## 10. Visualisasi

In [ ]:
fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(2, 3, figure=fig)
fig.suptitle(
    f'Pipeline 5 Fixed — Data Sintetis (+{len(synth_df)} pairs) + FocalLoss(γ={CFG.FOCAL_GAMMA}, α={CFG.FOCAL_ALPHA})',
    fontsize=14, fontweight='bold'
)

# Loss curve
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(history['tr_loss'], label='Train', color='#3498db', lw=2)
ax1.plot(history['vl_loss'], label='Val',   color='#e74c3c', lw=2)
ax1.set_title('Focal Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)

# AUC curve
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(history['tr_auc'], label='Train', color='#3498db', lw=2)
ax2.plot(history['vl_auc'], label='Val',   color='#e74c3c', lw=2)
ax2.axhline(y=best_auc, color='green', ls='--', alpha=0.7, label=f'Best={best_auc:.4f}')
ax2.axvline(x=CFG.WARMUP_EPOCHS, color='orange', ls=':', alpha=0.7, label=f'Unfreeze@{CFG.WARMUP_EPOCHS}')
ax2.set_title('AUC-ROC'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)

# F1 curve
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(history['tr_f1'], label='Train', color='#3498db', lw=2)
ax3.plot(history['vl_f1'], label='Val',   color='#e74c3c', lw=2)
ax3.set_title('Macro F1'); ax3.set_xlabel('Epoch'); ax3.legend(); ax3.grid(alpha=0.3)

# Confusion Matrix
ax4 = fig.add_subplot(gs[1, 0])
cm = confusion_matrix(best_labels_arr, final_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax4,
            xticklabels=CFG.CLASS_NAMES, yticklabels=CFG.CLASS_NAMES)
ax4.set_title('Confusion Matrix\n(Val: original only)')
ax4.set_xlabel('Predicted'); ax4.set_ylabel('True')

# ROC Curve
ax5 = fig.add_subplot(gs[1, 1])
fpr, tpr, _ = roc_curve(best_labels_arr, best_probs)
ax5.plot(fpr, tpr, color='#e74c3c', lw=2, label=f'AUC = {auc:.4f}')
ax5.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax5.set_title('ROC Curve')
ax5.set_xlabel('FPR (1 − Specificity)'); ax5.set_ylabel('TPR (Sensitivity)')
ax5.legend(); ax5.grid(alpha=0.3)

# Precision-Recall Curve
ax6 = fig.add_subplot(gs[1, 2])
prec_arr, rec_arr, _ = precision_recall_curve(best_labels_arr, best_probs)
ap = average_precision_score(best_labels_arr, best_probs)
ax6.plot(rec_arr, prec_arr, color='#8e44ad', lw=2, label=f'AP = {ap:.4f}')
baseline = best_labels_arr.mean()
ax6.axhline(y=baseline, color='gray', ls='--', alpha=0.5, label=f'Baseline = {baseline:.3f}')
ax6.set_title('Precision-Recall (Malignant)')
ax6.set_xlabel('Recall (Sensitivity)'); ax6.set_ylabel('Precision')
ax6.legend(); ax6.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/results_pipeline5_fixed.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot disimpan → results_pipeline5_fixed.png")

## 11. Summary — Pipeline 5 Fixed

In [ ]:
print("="*65)
print("  FINAL SUMMARY — Pipeline 5 Fixed: Data Sintetis + Focal Loss ⭐")
print("="*65)
print(f"  Model             : {CFG.MODEL_NAME}")
print(f"  Task              : Benign (BIRADS 1,2) vs Malignant (BIRADS 3,4,5)")
print(f"  Loss              : FocalLoss(gamma={CFG.FOCAL_GAMMA}, alpha={CFG.FOCAL_ALPHA})")
print(f"  Sampler           : Default shuffle")
print(f"  Sintetis source   : {CFG.SYNTH_ROOT}")
print(f"  Sintetis added    : {len(synth_df)} Malignant CC+MLO pairs (MAX_SYNTH=None)")
print(f"  Synth augment     : Agresif (Affine + Perspective + ColorJitter lebar)")
print(f"  Train Benign      : {n_orig_ben}  (original)")
print(f"  Train Malignant   : {n_orig_mal} (orig) + {len(synth_df)} (sintetis) = {n_orig_mal+len(synth_df)}")
print(f"  Train Total       : {len(train_dataset)}")
print(f"  Val set           : {len(val_df)} pairs  (original, TIDAK ada sintetis)")
print(f"  LR                : backbone={CFG.LR_BACKBONE}, head={CFG.LR_HEAD}")
print(f"  Scheduler         : ReduceLROnPlateau(mode=max, factor=0.5, patience=5)")
print(f"  Warmup            : {CFG.WARMUP_EPOCHS} epoch")
print(f"  Threshold         : {best_thresh_saved:.2f}")
print("-"*65)
print(f"  AUC-ROC           : {auc:.4f}")
print(f"  Accuracy          : {acc:.4f}")
print(f"  Macro F1          : {f1_macro:.4f}")
print(f"  Sensitivity       : {sensitivity:.4f}  ← recall malignant (prioritas utama)")
print(f"  Specificity       : {specificity:.4f}  ← recall benign")
print(f"  Precision         : {precision:.4f}")
print(f"  Kappa             : {kappa:.4f}")
print("="*65)
print("\nRingkasan semua pipeline:")
print("  P2 (BCE+weight)  : loss dikali pos_weight, data asli imbalanced")
print("  P3 (oversample)  : WeightedRandomSampler, data asli imbalanced")
print("  P4 (undersample) : Benign dipotong = jumlah Malignant")
print("  P5 original ⭐   : Sintetis 6400, CrossEntropyLoss (AUC=0.5798)")
print("  P5 fixed ⭐⭐    : Sintetis 6400, FocalLoss + aug agresif + scheduler fix")

metrics_p5 = pd.DataFrame([{
    'pipeline'        : 'P5_Fixed_FocalLoss',
    'loss'            : f'FocalLoss(gamma={CFG.FOCAL_GAMMA},alpha={CFG.FOCAL_ALPHA})',
    'sampler'         : 'default_shuffle',
    'synth_added'     : len(synth_df),
    'synth_augment'   : 'aggressive',
    'scheduler'       : 'ReduceLROnPlateau',
    'train_total'     : len(train_dataset),
    'auc'             : auc,
    'accuracy'        : acc,
    'macro_f1'        : f1_macro,
    'weighted_f1'     : f1_weight,
    'sensitivity'     : sensitivity,
    'specificity'     : specificity,
    'precision'       : precision,
    'kappa'           : kappa,
    'threshold'       : best_thresh_saved,
}])
metrics_p5.to_csv(f'{CFG.OUTPUT_DIR}/metrics_pipeline5_fixed.csv', index=False)
print("\nSaved → metrics_pipeline5_fixed.csv")